# Experimental Study

### Loading Libraries

In [2]:
%cd ../..

/Users/joaquinromero/Desktop


In [3]:
# Installing darts library to recreate experiments
!pip install darts

  Using cached pytorch_lightning-2.5.2-py3-none-any.whl.metadata (21 kB)
  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.2 MB/s eta 0:00:00
Using cached pytorch_lightning-2.5.2-py3-none-any.whl (825 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.6/547.6 kB 23.7 MB/s eta 0:00:00
Using cached slicer-0.0.8-py3-none-any.whl (15 kB)
  Attempting uninstall: pytorch-lightning
    Found existing installation: pytorch-lightning 2.5.4
    Uninstalling pytorch-lightning-2.5.4:
      Successfully uninstalled pytorch-lightning-2.5.4


In [ ]:
# Numerical Computing
import numpy as np

# Data Manipulation
import pandas as pd

# Data Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Notebook Optimizer
from tqdm.autonotebook import tqdm

# OS
import os
import random
import warnings
from pathlib import Path

# DARTS
from darts import TimeSeries
from darts.models import (
    NaiveSeasonal,
    ExponentialSmoothing,
    Theta, 
    FFT,
    LightGBMModel
)
from darts.metrics import mase
from darts.utils.utils import ModelMode, SeasonalityMode

In [ ]:
# Custom 
import plotly.figure_factory as ff
from src.utils.ts_utils import rmsse

In [ ]:
np.random.seed()

In [ ]:
os.makedirs("imgs/chapter_19", exist_ok=True)

preprocessed = Path.home() / "Desktop" / "data" / "london_smart_meters" / "preprocessed"

output = Path.home() / "Desktop" / "data" / "london_smart_meters" / "output"

In [ ]:
from darts.metrics.metrics import _get_values_or_raise


def rmsse(
    actual_series,
    pred_series,
    insample,
    m=1,
    intersect=True,
    *,
    reduction=np.mean,
):

    def _multivariate_mase(
        actual_series,
        pred_series,
        insample,
        m,
        intersect,
        reduction,
    ):

        assert (
            actual_series.width == pred_series.width
        ), "The two TimeSeries instances must have the same width."

        assert (
            actual_series.width == insample.width
        ), "The insample TimeSeries must have the same width as the other series."

        assert (
            insample.end_time() + insample.freq == pred_series.start_time()
        ), "The pred_series must be the forecast of the insample series"

        insample_ = (
            insample.quantile_timeseries(quantile=0.5)
            if insample.is_stochastic
            else insample
        )

        value_list = []
        for i in range(actual_series.width):
            # old implementation of mase on univariate TimeSeries
            y_true, y_hat = _get_values_or_raise(
                actual_series.univariate_component(i),
                pred_series.univariate_component(i),
                intersect,
                remove_nan_union=False,
            )

            x_t = insample_.univariate_component(i).values()
            errors = np.square(y_true - y_hat)
            scale = np.mean(np.square(x_t[m:] - x_t[:-m]))
            assert not np.isclose(scale, 0), "cannot use MASE with periodical signals"
            value_list.append(np.sqrt(np.mean(errors / scale)))

        return reduction(value_list)

    return _multivariate_mase(
        actual_series=actual_series,
        pred_series=pred_series,
        insample=insample,
        m=m,
        intersect=intersect,
        reduction=reduction,
    )

In [ ]:
from itertools import cycle

def format_plot(
    fig,
    legends=None,
    xlabel="Time",
    ylabel="Value",
    title="",
    font_size=15,
    title_font_size=20,
    width=900,
    height=500,
):
    # Renombrar trazas si se proveen leyendas
    if legends:
        names = cycle(legends)
        fig.for_each_trace(lambda t: t.update(name=next(names)))

    fig.update_layout(
        autosize=False,
        width=width,
        height=height,
        # Título (API moderna)
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
            yanchor="top",
            font=dict(size=title_font_size),
        ),
        legend_title_text=None,
        legend=dict(
            font=dict(size=font_size),
            orientation="h",
            yanchor="bottom",
            y=0.9,
            xanchor="right",
            x=1,
        ),
        # Ejes (API moderna)
        yaxis=dict(
            title=dict(text=ylabel, font=dict(size=font_size)),
            tickfont=dict(size=font_size),
        ),
        xaxis=dict(
            title=dict(text=xlabel, font=dict(size=font_size)),
            tickfont=dict(size=font_size),
        ),
        margin=dict(t=60, r=30, b=50, l=60),
        template="plotly_white",
    )
    return fig

In [ ]:
def format_y_pred(y_pred, name):
    y_pred = y_pred.data_array().to_series()
    y_pred.index = y_pred.index.get_level_values(0)
    y_pred.name = name
    return y_pred

### Reading Files

In [ ]:
# Reading the missing value imputed and train test split data
try:
    train_df = pd.read_parquet(preprocessed/"selected_blocks_train_missing_imputed.parquet")
    val_df = pd.read_parquet(preprocessed/"selected_blocks_val_missing_imputed.parquet")
    test_df = pd.read_parquet(preprocessed/"selected_blocks_test_missing_imputed.parquet")
except FileNotFoundError:
    display(HTML("""
    <div class="alert alert-block alert-warning">
    <b>Warning!</b> File not found. Please make sure you have run 01-Setting up Experiment Harness.ipynb in Chapter04
    </div>
    """))